# Final Project : Holistic Data Preparer


In [94]:
#All Imports
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder, Binarizer, KBinsDiscretizer, StandardScaler, Normalizer, MinMaxScaler, MaxAbsScaler, RobustScaler, FunctionTransformer, PowerTransformer
from sklearn.compose import ColumnTransformer


In [81]:
#Part B : Data Acquisition
df = pd.read_csv("D:\Data Preprocessing\Final_Project\customer_credit_risk_500.csv")
df.head()

<>:2: SyntaxWarning: invalid escape sequence '\D'
<>:2: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Admin\AppData\Local\Temp\ipykernel_11816\1383952276.py:2: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv("D:\Data Preprocessing\Final_Project\customer_credit_risk_500.csv")


,customer_id,age,gender,region,education_level,employment_type,annual_income,loan_amount,loan_purpose,credit_score,repayment_history,transaction_count,spending_ratio,join_date,default_flag
0,1001,59.0,Male,South,Secondary,Self-Employed,932180.0,273684,Education,674.0,4,43,59.05,2018-11-27,1
1,1002,22.0,NaN,South,Post-Graduate,Salaried,303355.0,116047,Car,621.0,0,68,51.49,2021-06-10,1
2,1003,NaN,Male,East,Post-Graduate,Salaried,1153277.0,586488,Home,663.0,4,70,60.82,2022-12-31,1
3,1004,29.0,NaN,South,Graduate,NaN,1453617.0,304648,Car,729.0,6,79,27.32,2022-08-26,1
4,1005,28.0,NaN,North,Post-Graduate,Salaried,1973415.0,898088,Business,653.0,5,82,70.54,2019-06-20,1


In [82]:
#Part C : Data Understanding & Cleaning

#Exloring Dataset
print("Dataset Information")
print(df.info,"\n")
print("Dataset Description")
print(df.describe(),"\n")

#Pandas Profiling
profile = ProfileReport(df=df, explorative=True)
profile.to_file("Report_Before_Analysis")

Dataset Information
<bound method DataFrame.info of      customer_id   age  gender region education_level employment_type  \
0           1001  59.0    Male  South       Secondary   Self-Employed   
1           1002  22.0     NaN  South   Post-Graduate        Salaried   
2           1003   NaN    Male   East   Post-Graduate        Salaried   
3           1004  29.0     NaN  South        Graduate             NaN   
4           1005  28.0     NaN  North   Post-Graduate        Salaried   
..           ...   ...     ...    ...             ...             ...   
495         1496  46.0  Female   East   Post-Graduate   Self-Employed   
496         1497  53.0  Female   West       Secondary      Unemployed   
497         1498  43.0     NaN  South         Primary             NaN   
498         1499  36.0     NaN  North       Secondary        Salaried   
499         1500   NaN  Female   West       Secondary   Self-Employed   

     annual_income  loan_amount loan_purpose  credit_score  repayment_h

Render HTML: 100%|██████████| 1/1 [00:00<00:00,  4.15it/s]
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\ydata_profiling\profile_report.py:386: UserWarning: Extension  not supported. For now we assume .html was intended. To remove this warning, please use .html or .json.
  warnings.warn(
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 94.41it/s]


In [83]:
#Handle Missing Data

#Simple Imputer Numeric 
df['annual_income']= SimpleImputer(strategy='mean').fit_transform(df[['annual_income']])

#Simple Imputer Category and Most frequent value
df['gender'] = SimpleImputer(strategy='most_frequent').fit_transform(df[['gender']]).flatten()

#Missing Indicator + Random Sample Imputation.
df['missing_value'] = df['credit_score'].isnull().astype(int)
random_sample = df['credit_score'].dropna().sample(df['credit_score'].isnull().sum(), random_state=42)
random_sample.index = df[df['credit_score'].isnull()].index
df.loc[df['credit_score'].isnull(), 'credit_score'] = random_sample

#KNN imputer
df['age']=KNNImputer(n_neighbors=3).fit_transform(df[['age']])
df['loan_amount']=KNNImputer(n_neighbors=3).fit_transform(df[['loan_amount']])

#MICE
cols = ['annual_income', 'credit_score']
mice = IterativeImputer(random_state=42).fit_transform(df[cols])

#Drop Rows
df_drop=df.dropna(subset=['employment_type'])

In [84]:
#Part D : Outlier Handling

#Z-Score Method

#mean of credit score
df_cs_m = df['credit_score'].mean()
print("Mean of Credit Score: ", df_cs_m, "\n")

#standard deviation of credit score
df_cs_s = df['credit_score'].std()
print("Standard Deviation of Credit Score: ", df_cs_s, "\n")

#z-score of credit score
z_cs = (df['credit_score']-df_cs_m)/df_cs_s

#Identify Outliers
df_clean_cs = df[(z_cs > -3) & (z_cs < 3)]

#mean of age
df_age_m = df['age'].mean()
print("Mean of age: ", df_age_m, "\n")

#standard deviation of age
df_age_s = df['age'].std()
print("Standard Deviation of age: ", df_age_s, "\n")

#z-score of age
z_age = (df['age']-df_age_m)/df_age_s

#Identify Outliers
df_clean_age = df[(z_age > -3) & (z_age < 3)]

Mean of Credit Score:  720.92 

Standard Deviation of Credit Score:  70.83276191368398 

Mean of age:  39.69642857142856 

Standard Deviation of age:  10.905011100665595 



In [85]:
#IQR Method

#Calculate Q1 and Q3 for annual income
q25_la = np.quantile(df['loan_amount'], 0.25)
q75_la = np.quantile(df['loan_amount'], 0.75)

#Calculate IQR
iqr_la = q75_la - q25_la

#Calculate lower and upper bounds
lower_bound_la = q25_la - 1.5 * iqr_la
upper_bound_la = q75_la + 1.5 * iqr_la

#Identify Outliers
df_clean_la = df[(df['loan_amount'] >= lower_bound_la) & (df['loan_amount'] <= upper_bound_la)]

print("Shape Before Outliers : ", df['loan_amount'].shape[0])
print("Shape After Outliers : ", df_clean_la['loan_amount'].shape[0],"\n")


#Percentile 

#Calculate 1st and 99th percentiles for annual income
percentile_1_tc = np.percentile(df['transaction_count'], 1)
percentile_99_tc = np.percentile(df['transaction_count'], 99)

df_clean_tc = df[(df['transaction_count']>=percentile_1_tc) & (df['transaction_count']<=percentile_99_tc)]


#Winsorization Technique
df_win = df.copy()

#calculate 5th and 95th percentiles for annual income
percentile_5_la = np.percentile(df['loan_amount'], 5)
percentile_95_la = np.percentile(df['loan_amount'], 95)

#Apply Winsorization
df_win['loan_amount'] = df_win['loan_amount'].clip(lower=percentile_5_la, upper=percentile_95_la)

print("Original Mean:", df['loan_amount'].mean())
print("Winsorized Mean:", df_win['loan_amount'].mean(),"\n")

print("Original Shape:", df.shape)
print("Winsorized Shape:", df_win.shape)

Shape Before Outliers :  500
Shape After Outliers :  495 

Original Mean: 523072.692
Winsorized Mean: 517949.406 

Original Shape: (500, 16)
Winsorized Shape: (500, 16)


In [88]:
#Part E : Feature Engineering

#Mixed variables
num_cols = ['age', 'annual_income', 'loan_amount', 'credit_score', 'transaction_count', 'spending_ratio']
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')

cat_cols = ['gender', 'region', 'education_level', 'employment_type', 'loan_purpose']
df[cat_cols] = df[cat_cols].astype('category')

#Date time Extract 
df['join_date'] = pd.to_datetime(df['join_date'])
df['year'] = df['join_date'].dt.year
df['month'] = df['join_date'].dt.month
df['day'] = df['join_date'].dt.day
df['weekday'] = df['join_date'].dt.weekday

#Ordinal Encoding
ordinal_encoder = OrdinalEncoder().fit_transform(df[['education_level']])

#Label Encoding
label_encoder = LabelEncoder().fit_transform(df['gender'])

#One-Hot Encoding
one_hot_encoder = OneHotEncoder().fit_transform(df[['region']]).toarray()
one_hot_encoder = OneHotEncoder().fit_transform(df[['loan_purpose']]).toarray()

#Binning
df['annual_income_bin'] = pd.cut(df['annual_income'], bins=[0, 300000, 600000, 900000, 2000000], labels=['low', 'medium', 'high','very high'])

#Binarization 
df["credit_score_binarize"] = Binarizer(threshold=600).fit_transform(df[['credit_score']]).flatten()

df.head()
#Quantile Binning
df['spending_ratio_bin'] = pd.qcut(df['spending_ratio'].rank(method='first'), q=4, labels=['level 1', 'level 2', 'level 3', 'level 4'])

#K-means binning
df['loan_amount_kbins']=KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='kmeans').fit_transform(df[['loan_amount']]).flatten()

In [ ]:
#Part F : Feature Scaling

#Standard Scaler
df['annual_income_scaled'] = StandardScaler().fit_transform(df[['annual_income']])

#Min-Max Scaler
df['transaction_count_scaled']=MinMaxScaler().fit_transform(df[['transaction_count']])

#Normalizer
df['spending_ratio_normalized']=Normalizer().fit_transform(df[['spending_ratio']])

#MaxAbs Scaler
df['credit_score_maxabs']=MaxAbsScaler().fit_transform(df[['credit_score']])

#Robust Scaler
df['age_robust']=RobustScaler().fit_transform(df[['age']])

In [ ]:
#Part G : Feature Construction & Transformation 

#Function Transformer 
df['loan_sqrt'] = FunctionTransformer(np.sqrt).fit_transform(df[['loan_amount']])

#Power Transformer
df[['income_power', 'loan_power']]= PowerTransformer(method="yeo-johnson").fit_transform(df[['annual_income', 'loan_amount']])

#Column Transformer
ct = ColumnTransformer([('num', StandardScaler(), ['annual_income', 'loan_amount']),('cat', OneHotEncoder(sparse_output=False), ['region'])]).fit_transform(df)

#Feature Construction

df['debt_to_income'] = df['loan_amount'] / df['annual_income']
df['avg_monthly_txn'] = df['transaction_count'] / df['month']
df['spending_to_income'] = df['spending_ratio'] / df['annual_income']

In [110]:
#Final Dataset
df.to_csv("new_holistic_data.csv", index=False)

### Data Preprocessing Report
- Missing Values: Used mean/median, most frequent, KNN, MICE, and random imputation. Advanced methods preserved data patterns better.
- Outliers: Handled using Z-score, IQR, and Winsorization to reduce extreme values without losing data.
- Encoding: Applied ordinal (education), label (gender), and one-hot (region, loan purpose).
- Scaling/Transformations: Used standard, min-max, robust scaling, and log/power transforms to normalize data and reduce skewness.
- Feature Engineering: Created debt-to-income, avg monthly transactions, and spending-to-income ratios to improve insights.
- Final Dataset: Clean, structured, and ready for ML modeling.